# 08 · Análisis geoespacial y resultados
| Sección | Tipo de dato | Resultado |
|---|---|---|
| 1. Superficies | SUPERFICIE | frecuencia de SCM, Kc con/sin SCM, pérdida relativa y **déficit energético (%)** en GeoTIFF |
| 2. Municipios | ÁREA | estadística zonal, **I de Moran**, **LISA** y **Getis-Ord Gi\*** |
| 3. Relieve | SUPERFICIE | déficit y frecuencia de SCM por bandas de altitud |
| 4. Parques (estimado) | PUNTO | Kc según distancia al SCM y etapa del ciclo de vida |
| 5. Validación y pérdida real | PUNTO | estimado vs. XM y **pérdida real** de factor de planta bajo SCM |

Todo se guarda en la carpeta `resultados/`.

In [1]:
# ▶ CELDA DE PREPARACIÓN — ejecútala siempre de primera
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    # En Colab los archivos se guardan en tu Google Drive para no perderlos
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/scm_magdalena")   # cámbiala si quieres
    CARPETA.mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA)
    PAQUETES = ["numpy", "scipy", "pandas", "xarray", "dask", "netCDF4", "h5netcdf", "s3fs", "pyproj", "shapely", "geopandas", "rasterio", "pvlib", "pydataxm", "libpysal", "esda", "matplotlib"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PAQUETES], check=True)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Carpeta de trabajo:", os.getcwd())
assert Path("config.py").exists(), "No encuentro config.py: ejecuta primero el notebook 00 en esta misma carpeta"

Carpeta de trabajo: c:\Users\User\Desktop\Unal\Analisis Geoespacial\notebooks_scm_magdalena


In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.transform import from_origin
import matplotlib.pyplot as plt
import libpysal
from esda.moran import Moran, Moran_Local
from esda.getisord import G_Local
import config as C

TRANSF = from_origin(C.DOM_ANALISIS[2], C.DOM_ANALISIS[1], C.RES_MALLA, C.RES_MALLA)
EXT = [C.DOM_ANALISIS[2], C.DOM_ANALISIS[3], C.DOM_ANALISIS[0], C.DOM_ANALISIS[1]]
mpios = gpd.read_file(C.PROC / "municipios.gpkg").reset_index(drop=True)
parques = gpd.read_file(C.PROC / "parques.gpkg")

def guardar_tif(nombre, arr):
    with rasterio.open(C.OUT / f"{nombre}.tif", "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1],
                       count=1, dtype="float32", crs="EPSG:4326", transform=TRANSF, nodata=np.nan,
                       compress="lzw") as dst:
        dst.write(arr.astype("float32"), 1)

c:\Users\User\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Superficies de pérdida
- **Frecuencia diurna de SCM:** fracción de horas de sol con un SCM encima.
- **Pérdida relativa:** $1 - \overline{K_c}_{con\,SCM} / \overline{K_c}_{sin\,SCM}$.
- **Déficit energético (%):** energía que no llegó por estar bajo un SCM, comparada con la que habría llegado con el $K_c$ típico sin SCM *a esa misma hora*, dividida por la energía total de referencia.

In [ ]:
irr = xr.open_mfdataset(sorted(C.PROC.glob("irradiancia_*.nc")))
mask = xr.open_dataset(C.PROC / "scm_mascara.nc")["scm_id"]
t = np.intersect1d(irr.time.values, mask.time.values)
kc = irr["kc"].sel(time=t).values
gcs = irr["ghi_despejado"].sel(time=t).values
con = mask.sel(time=t).values > 0
dia = np.isfinite(kc)
with np.errstate(invalid="ignore", divide="ignore"):
    frec = (con & dia).sum(0) / dia.sum(0)
    kc_con = np.nanmean(np.where(con, kc, np.nan), axis=0)
    kc_sin = np.nanmean(np.where(~con, kc, np.nan), axis=0)
    perdida = 1 - kc_con / kc_sin
    horas = pd.DatetimeIndex(t).hour
    kc_ref = np.empty_like(kc)
    for h in np.unique(horas):
        s = horas == h
        kc_ref[s] = np.nanmean(np.where(~con[s], kc[s], np.nan), axis=0)
    e_ref = np.nansum(kc_ref * gcs, axis=0)
    e_perdida = np.nansum(np.where(con, kc_ref * gcs - kc * gcs, 0), axis=0)
    deficit = 100 * e_perdida / e_ref

sup = {"frecuencia_scm_diurna": frec, "kc_medio_con_scm": kc_con, "kc_medio_sin_scm": kc_sin,
       "perdida_relativa": perdida, "deficit_energetico": deficit}
for k, v in sup.items():
    guardar_tif(k, v)
print("GeoTIFF guardados en", C.OUT)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(19, 7))
for ax, (k, titulo, cmap, f) in zip(axs, [
        ("frecuencia_scm_diurna", "Frecuencia diurna de SCM (%)", "Blues", 100),
        ("perdida_relativa", "Pérdida relativa de Kc bajo SCM (%)", "OrRd", 100),
        ("deficit_energetico", "Déficit de energía solar por SCM (%)", "YlOrRd", 1)]):
    im = ax.imshow(f * sup[k], extent=EXT, cmap=cmap)
    plt.colorbar(im, ax=ax, shrink=0.75)
    mpios.boundary.plot(ax=ax, color="k", linewidth=0.3)
    parques.plot(ax=ax, color="#E0A526", edgecolor="k", marker="^", markersize=100)
    ax.set_title(titulo)
plt.tight_layout(); plt.savefig(C.OUT / "mapas_superficies.png", dpi=150); plt.show()

In [ ]:
# Ciclo diurno: fracción del Magdalena bajo SCM (24 h, canal 13)
hl = (pd.DatetimeIndex(mask.time.values) - pd.Timedelta(hours=5)).hour
ciclo = pd.Series((mask.values > 0).mean(axis=(1, 2)), index=hl).groupby(level=0).mean()
ax = (100 * ciclo).plot(kind="bar", figsize=(10, 4), color="#3D6E8F")
ax.axvspan(5.5, 18.5, color="#E0A526", alpha=0.12, label="Horas de sol")
ax.set_xlabel("Hora local"); ax.set_ylabel("% del dominio bajo SCM"); ax.legend()
plt.tight_layout(); plt.savefig(C.OUT / "ciclo_diurno_scm.png", dpi=150); plt.show()

## 2. Estadística zonal por municipio y autocorrelación espacial
- **I de Moran global:** ¿el déficit se agrupa en el espacio?
- **LISA:** clústeres Alto-Alto, Bajo-Bajo y atípicos.
- **Getis-Ord Gi\*:** puntos calientes y fríos.

La vecindad es de tipo reina (*Queen*): municipios que comparten borde o vértice.

In [ ]:
ids = features.rasterize(((g, i + 1) for i, g in enumerate(mpios.geometry)),
                         out_shape=frec.shape, transform=TRANSF, fill=0, all_touched=True)
for k, v in sup.items():
    d = pd.DataFrame({"id": ids.ravel(), "v": v.ravel()}).query("id > 0")
    medias = d.groupby("id")["v"].mean()
    mpios[k] = mpios.index.map(lambda i: medias.get(i + 1, np.nan))

var = "deficit_energetico"
mp = mpios.dropna(subset=[var]).copy()
w = libpysal.weights.Queen.from_dataframe(mp, use_index=False)
w.transform = "r"
mi = Moran(mp[var].values, w)
print(f"I de Moran ({var}) = {mi.I:.3f}   p (permutaciones) = {mi.p_sim:.3f}")
lisa = Moran_Local(mp[var].values, w)
cuad = {1: "Alto-Alto", 2: "Bajo-Alto", 3: "Bajo-Bajo", 4: "Alto-Bajo"}
mp["lisa"] = [cuad[q] if p < 0.05 else "No significativo" for q, p in zip(lisa.q, lisa.p_sim)]
gi = G_Local(mp[var].values, w, star=True)
mp["gi_z"], mp["gi_p"] = gi.Zs, gi.p_sim
mp.to_file(C.OUT / "municipios_resultados.gpkg", driver="GPKG")
pd.DataFrame({"I_Moran": [mi.I], "p": [mi.p_sim]}).to_csv(C.OUT / "moran.csv", index=False)
col_nombre = "shapeName" if "shapeName" in mp.columns else mp.columns[0]
mp[[col_nombre, var, "lisa", "gi_z"]].sort_values(var, ascending=False).round(2)

In [ ]:
colores_lisa = {"Alto-Alto": "#B2182B", "Bajo-Bajo": "#2166AC", "Alto-Bajo": "#F4A582",
                "Bajo-Alto": "#92C5DE", "No significativo": "#E5E5E5"}
fig, axs = plt.subplots(1, 3, figsize=(20, 7))
mp.plot(var, ax=axs[0], legend=True, cmap="YlOrRd", edgecolor="0.5", linewidth=0.3)
axs[0].set_title("Déficit de energía solar por SCM (%)")
mp.plot(color=mp["lisa"].map(colores_lisa), ax=axs[1], edgecolor="0.5", linewidth=0.3)
for etiqueta, c in colores_lisa.items():
    axs[1].scatter([], [], color=c, label=etiqueta, s=80)
axs[1].legend(loc="lower left"); axs[1].set_title(f"LISA (I de Moran = {mi.I:.2f}, p = {mi.p_sim:.3f})")
mp.plot("gi_z", ax=axs[2], legend=True, cmap="RdBu_r", edgecolor="0.5", linewidth=0.3)
axs[2].set_title("Puntos calientes (Getis-Ord Gi*, z)")
for ax in axs:
    parques.plot(ax=ax, color="k", marker="^", markersize=60); ax.set_axis_off()
plt.tight_layout(); plt.savefig(C.OUT / "mapa_municipios.png", dpi=150); plt.show()

## 3. Relieve: pérdida y frecuencia de SCM según la altitud

In [ ]:
with rasterio.open(C.PROC / "dem_2km.tif") as src:
    z = src.read(1)
d = pd.DataFrame({"z": z.ravel(), "frecuencia_%": 100 * frec.ravel(), "deficit_%": deficit.ravel()}).dropna()
d["banda"] = pd.cut(d.z, [-50, 100, 250, 500, 1000, 2000, 3000, 6000])
res = d.groupby("banda", observed=True)[["frecuencia_%", "deficit_%"]].mean()
res["n_pixeles"] = d.groupby("banda", observed=True).size()
res.to_csv(C.OUT / "scm_vs_altitud.csv")
ax = res[["frecuencia_%", "deficit_%"]].plot(kind="bar", figsize=(10, 4), color=["#3D6E8F", "#E0A526"])
ax.set_xlabel("Altitud (m s.n.m.)"); ax.set_ylabel("%")
plt.tight_layout(); plt.savefig(C.OUT / "deficit_vs_altitud.png", dpi=150); plt.show()
res.round(2)

## 4. Parques: Kc estimado según distancia al SCM y etapa del ciclo de vida

In [ ]:
fv = pd.read_csv(C.PROC / "fv_parques_horario.csv").dropna(subset=["kc"])
etiquetas = ["bajo SCM", "0–25", "25–50", "50–100", "100–200", ">200 km"]
fv["rango_dist"] = pd.cut(fv.dist_scm_km, [-1, 0, 25, 50, 100, 200, np.inf], labels=etiquetas)
tab = fv.groupby(["parque", "rango_dist"], observed=True)["kc"].mean().unstack(0)
tab.to_csv(C.OUT / "kc_por_distancia_scm.csv")
et = fv[fv.bajo_scm].groupby(["parque", "etapa"])["kc"].mean().unstack(0)
et.to_csv(C.OUT / "kc_por_etapa_scm.csv")

fig, axs = plt.subplots(1, 2, figsize=(15, 4.5))
tab.plot(ax=axs[0], marker="o"); axs[0].set_ylabel("Kc medio"); axs[0].set_xlabel("Distancia al SCM más cercano")
axs[0].set_title("Kc según distancia al SCM")
et.reindex(["iniciación", "madurez", "disipación"]).plot(kind="bar", ax=axs[1])
axs[1].set_ylabel("Kc medio bajo SCM"); axs[1].set_title("Kc según etapa del SCM")
plt.tight_layout(); plt.savefig(C.OUT / "kc_distancia_etapa.png", dpi=150); plt.show()
display(tab.round(3)); display(et.round(3))

## 5. Validación con XM y pérdida REAL
1. **Validación:** sesgo, RMSE y correlación entre la generación estimada (GOES + PVWatts) y la real (XM).
2. **Pérdida real:** factor de planta bajo SCM frente a sin SCM, comparando la **misma hora del día** y ponderando por el número de horas con SCM.

XM reporta la energía del intervalo [t, t+1 h). GOES es instantáneo, así que la estimación se promedia entre t y t+1 h.

In [ ]:
xm = pd.read_csv(C.PROC / "xm_generacion_horaria.csv", parse_dates=["time"])
fv = pd.read_csv(C.PROC / "fv_parques_horario.csv")
fv["time"] = pd.to_datetime(fv["time"], utc=True).dt.tz_convert(None)
cap = pd.read_csv(C.ARCHIVO_PARQUES).set_index("nombre")["capacidad_mw"]
filas, unidos = [], []

for parque, f in fv.groupby("parque"):
    f = f.set_index("time").sort_index()
    est = (f["p_ac_mw"] + f["p_ac_mw"].shift(-1, freq="h").reindex(f.index)) / 2
    d = xm[xm.parque == parque].set_index("time").join(
        pd.DataFrame({"estimado_mwh": est, "bajo_scm": f["bajo_scm"].astype(bool),
                      "dist_scm_km": f["dist_scm_km"], "etapa": f["etapa"]}), how="inner")
    d = d.dropna(subset=["energia_mwh", "estimado_mwh"])
    if d.empty:
        print(f"{parque}: sin datos de XM en común"); continue
    d["parque"] = parque
    d["factor_planta"] = d["energia_mwh"] / cap[parque]
    d["hora_local"] = (d.index - pd.Timedelta(hours=5)).hour
    err = d["estimado_mwh"] - d["energia_mwh"]
    por_hora = d.groupby(["hora_local", "bajo_scm"])["factor_planta"].mean().unstack()
    n_con = d[d.bajo_scm].groupby("hora_local").size()
    perdida = np.nan
    if True in por_hora and False in por_hora:
        r = (1 - por_hora[True] / por_hora[False]).dropna()
        wgt = n_con.reindex(r.index).fillna(0)
        perdida = 100 * float((r * wgt).sum() / wgt.sum()) if wgt.sum() > 0 else np.nan
    filas.append({"parque": parque, "n_horas": len(d), "n_horas_bajo_scm": int(n_con.sum()),
                  "sesgo_mwh": err.mean(), "rmse_mwh": np.sqrt((err ** 2).mean()),
                  "r": d["estimado_mwh"].corr(d["energia_mwh"]), "perdida_real_%": perdida})
    unidos.append(d)

res = pd.DataFrame(filas)
res.to_csv(C.OUT / "validacion_y_perdida_real.csv", index=False)
res.round(3)

In [ ]:
todo = pd.concat(unidos)
fig, axs = plt.subplots(1, 3, figsize=(20, 5.5))
for parque, d in todo.groupby("parque"):
    c = np.where(d.bajo_scm, "#B5810F", "#3D6E8F")
    axs[0].scatter(d["energia_mwh"], d["estimado_mwh"], s=6, alpha=0.4, c=c)
lim = [0, todo[["energia_mwh", "estimado_mwh"]].max().max()]
axs[0].plot(lim, lim, "k--", lw=1)
axs[0].set_xlabel("Generación real XM (MWh)"); axs[0].set_ylabel("Estimada desde GOES (MWh)")
axs[0].set_title("Validación (naranja = bajo SCM)")

todo.groupby(["hora_local", "bajo_scm"])["factor_planta"].mean().unstack() \
    .rename(columns={False: "sin SCM", True: "bajo SCM"}).plot(ax=axs[1], marker="o", color=["#3D6E8F", "#B5810F"])
axs[1].set_xlabel("Hora local"); axs[1].set_ylabel("Factor de planta"); axs[1].set_title("Ciclo diurno real con y sin SCM")

todo["rango_dist"] = pd.cut(todo.dist_scm_km, [-1, 0, 25, 50, 100, 200, np.inf], labels=etiquetas)
fpd = todo.groupby(["parque", "rango_dist"], observed=True)["factor_planta"].mean().unstack(0)
fpd.plot(ax=axs[2], marker="o"); axs[2].set_ylabel("Factor de planta medio")
axs[2].set_xlabel("Distancia al SCM más cercano"); axs[2].set_title("Factor de planta real según distancia")
plt.tight_layout(); plt.savefig(C.OUT / "validacion_y_perdida_real.png", dpi=150); plt.show()

fpd.to_csv(C.OUT / "factor_planta_por_distancia_scm.csv")
todo[todo.bajo_scm].groupby(["parque", "etapa"])["factor_planta"].mean().unstack(0) \
    .to_csv(C.OUT / "factor_planta_por_etapa_scm.csv")

## Resumen de productos en `resultados/`
| Archivo | Para qué sección del artículo |
|---|---|
| `mapa_area_estudio.png`, `mapa_dem.png` | Datos y metodología |
| `scm_descriptivos.png`, `mapa_frecuencia_scm.png`, `ciclo_diurno_scm.png` | Resultados: climatología de SCM |
| `mapas_superficies.png` y los GeoTIFF | Resultados: superficies de pérdida |
| `mapa_municipios.png`, `moran.csv`, `municipios_resultados.gpkg` | Resultados: análisis por áreas |
| `deficit_vs_altitud.png` | Discusión: papel del relieve |
| `validacion_y_perdida_real.png` / `.csv` | Resultados: validación y pérdida real |

In [ ]:
for f in sorted(C.OUT.iterdir()):
    print(f.name)